In [4]:
"""
Neural Atom Prevalence (NAP) — YearPredictionMSD  (UCI Regression)
===================================================================
Adapted from the MNIST classification pipeline to regression following
the paper's Section B.2–B.3 hyperparameters for the Year dataset.

Architecture  : 90 → 100 (BayesianLinear + Swish) → 2 (mean, log σ)
                One hidden layer, heteroscedastic Gaussian output.
Task          : Predict year of song release (regression, RMSE metric)
Train/Test    : 9 : 1 random split (seeded)
Batch size    : 256            (paper B.3)
Epochs        : 100 per phase  (paper B.3)
LR            : 1e-3           (paper B.3)
MC samples S  : 1 during train (paper B.3), 10–20 for evaluation
Init μ        : U(−0.6, 0.6)  (paper B.2)
Init ρ        : −6 → σ ≈ 0    (paper B.2)
Init γ (π̃)  : 0.99           (paper B.2)

Pipeline phases (same 5-phase structure as MNIST):
  Phase 1 : Bayesian Lottery Ticket — IMP (5 rounds, 20% prune/round)
  Phase 2 : SS-IG on winning ticket with soft Gumbel-Softmax
  Phase 3 : Poisson-Binomial optimal layer-size selection
  Phase 4 : Neural atom selection by ranked posterior inclusion prob.
  Phase 5 : Fine-tuning the sparse BNN from epoch-0 weights

Outputs saved to disk:
  nap_year_sparse_model.pt       — model checkpoint + training histories
  nap_year_training_curves.png   — 6-panel diagnostic figure
"""

import os
import math
import copy
import random
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from scipy import stats as scipy_stats

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

# ── reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_WORKERS = 4      # fewer workers — tabular data loads fast


# ─────────────────────────────────────────────────────────────────────────────
# 1.  DATA LOADING — YearPredictionMSD
# ─────────────────────────────────────────────────────────────────────────────

def get_year_loaders(data_path: str,
                     batch_size: int  = 256,
                     train_frac: float = 0.9) -> tuple[DataLoader, DataLoader,
                                                        float, float, float, float]:
    """
    Load YearPredictionMSD from a local text/CSV file (no header, space/comma
    separated).  Column 0 is the target (year), columns 1–90 are features.

    Returns
    -------
    train_loader, test_loader,
    y_mean, y_std   — target normalisation constants (for back-transforming RMSE)
    X_mean, X_std   — feature normalisation (stored for diagnostics)
    """
    print(f"[data] loading YearPredictionMSD from: {data_path}")
    df = pd.read_csv(data_path, header=None, sep=None, engine="python")
    print(f"[data] raw shape: {df.shape}")

    y = df.iloc[:, 0].values.astype(np.float32)   # year
    X = df.iloc[:, 1:].values.astype(np.float32)  # 90 audio features

    # ── feature standardisation (zero-mean, unit-variance) ───────────────────
    X_mean = X.mean(axis=0)
    X_std  = X.std(axis=0) + 1e-8
    X_norm = (X - X_mean) / X_std

    # ── target standardisation ────────────────────────────────────────────────
    y_mean = float(y.mean())
    y_std  = float(y.std())
    y_norm = (y - y_mean) / y_std

    # ── 9:1 random split (reproducible) ─────────────────────────────────────
    n       = len(y)
    n_train = int(n * train_frac)
    rng     = np.random.RandomState(SEED)
    idx     = rng.permutation(n)
    tr_idx  = idx[:n_train]
    te_idx  = idx[n_train:]

    X_tr, y_tr = X_norm[tr_idx], y_norm[tr_idx]
    X_te, y_te = X_norm[te_idx], y_norm[te_idx]

    def _make_loader(X, y, shuffle):
        ds = TensorDataset(
            torch.from_numpy(X),
            torch.from_numpy(y).unsqueeze(1)   # (N,1) — easier broadcasting
        )
        return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                          num_workers=N_WORKERS, pin_memory=True,
                          persistent_workers=(N_WORKERS > 0))

    train_loader = _make_loader(X_tr, y_tr, shuffle=True)
    test_loader  = _make_loader(X_te, y_te, shuffle=False)

    print(f"[data] train={n_train:,}  test={n-n_train:,} "
          f"| batch={batch_size} | device={DEVICE}")
    print(f"[data] year range: {y.min():.0f}–{y.max():.0f} "
          f"| y_mean={y_mean:.2f}  y_std={y_std:.2f}")
    print(f"[data] features: {X.shape[1]}")

    return (train_loader, test_loader,
            y_mean, y_std,
            X_mean.tolist(), X_std.tolist())


# ─────────────────────────────────────────────────────────────────────────────
# 2.  NETWORK ARCHITECTURE — one hidden layer, regression
# ─────────────────────────────────────────────────────────────────────────────

class SwishActivation(nn.Module):
    """f(x) = x · σ(x)  (SiLU — avoids dying-neuron problem)."""
    def forward(self, x):
        return x * torch.sigmoid(x)


class BayesianLinear(nn.Module):
    """
    Variational Bayesian linear layer with spike-and-slab prior.

    Initialisation follows paper Appendix B.2:
      μ  ~ U(−0.6, 0.6)   (paper: uniform ±0.6)
      ρ  = −6              → σ = softplus(−6) ≈ 0.0025  (weights stay near 0)
      γ  = 0.99            → logit_π ≈ 4.595           (fully connected at t=0)
    """
    def __init__(self, in_features: int, out_features: int,
                 sigma0: float = 1.0):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
        self.sigma0       = sigma0

        # weight + bias concatenated: (out, in+1)
        self.mu       = nn.Parameter(torch.empty(out_features, in_features + 1))
        self.rho      = nn.Parameter(torch.empty(out_features, in_features + 1))
        self.logit_pi = nn.Parameter(torch.empty(out_features))
        self._init_params()

    def _init_params(self):
        """Paper B.2: μ ~ U(−0.6, 0.6), ρ = −6, γ = 0.99 → logit(0.99)."""
        nn.init.uniform_(self.mu, -0.6, 0.6)
        nn.init.constant_(self.rho, -6.0)
        nn.init.constant_(self.logit_pi, math.log(0.99 / 0.01))   # ≈ 4.595

    # ── epoch-0 snapshot for LTH rewinding ───────────────────────────────────
    def snapshot_epoch0(self):
        self._mu0       = self.mu.data.clone()
        self._rho0      = self.rho.data.clone()
        self._logit_pi0 = self.logit_pi.data.clone()

    def rewind_to_epoch0(self):
        self.mu.data.copy_(self._mu0)
        self.rho.data.copy_(self._rho0)
        self.logit_pi.data.copy_(self._logit_pi0)

    @property
    def pi_tilde(self) -> torch.Tensor:
        return torch.sigmoid(self.logit_pi)

    def gumbel_softmax_gate(self, tau: float = 0.5) -> torch.Tensor:
        alpha  = torch.zeros_like(self.logit_pi).uniform_().clamp(1e-6, 1-1e-6)
        gumbel = torch.log(alpha) - torch.log(1 - alpha)
        return torch.sigmoid((self.logit_pi + gumbel) / tau)

    def forward(self, x: torch.Tensor, tau: float = 0.5,
                hard: bool = False) -> torch.Tensor:
        sigma    = F.softplus(self.rho)
        eps      = torch.randn_like(self.mu)
        w_sample = self.mu + sigma * eps
        w        = w_sample[:, :-1]
        b        = w_sample[:, -1]
        v        = ((self.pi_tilde > 0.5).float()
                    if hard else self.gumbel_softmax_gate(tau))
        return F.linear(x, w, b) * v.unsqueeze(0)

    # ── KL terms ──────────────────────────────────────────────────────────────
    def kl_v(self, prior_pi: float) -> torch.Tensor:
        pi = self.pi_tilde.clamp(1e-6, 1 - 1e-6)
        p  = torch.tensor(prior_pi, device=pi.device).clamp(1e-6, 1-1e-6)
        return (pi * (pi.log() - p.log()) +
                (1-pi) * ((1-pi).log() - (1-p).log())).sum()

    def kl_weights(self) -> torch.Tensor:
        sigma  = F.softplus(self.rho)
        sigma0 = torch.tensor(self.sigma0, device=self.mu.device)
        kl_j   = 0.5 * ((sigma/sigma0).pow(2) + (self.mu/sigma0).pow(2)
                         - 1 - 2*(sigma/sigma0).log()).sum(dim=1)
        return (self.pi_tilde * kl_j).sum()


class NAPNetworkYear(nn.Module):
    """
    One-hidden-layer BNN for regression:  D → 100 → 2  (mean, log σ).

    The output head produces two values per sample:
      out[:, 0]  =  predicted mean   μ̂
      out[:, 1]  =  log predicted σ  → σ̂ = exp(log σ)

    This heteroscedastic parameterisation lets the model learn input-dependent
    noise, which is essential for UQ decomposition.
    """
    def __init__(self, input_dim: int = 90, hidden: int = 100,
                 sigma0: float = 1.0):
        super().__init__()
        self.input_dim = input_dim
        self.hidden    = hidden
        self.layer1    = BayesianLinear(input_dim, hidden, sigma0)
        self.out       = nn.Linear(hidden, 2)      # [mean, log_sigma]
        self.act       = SwishActivation()
        self._reset_output_head()

    def _reset_output_head(self):
        nn.init.kaiming_uniform_(self.out.weight, a=math.sqrt(5))
        nn.init.zeros_(self.out.bias)

    def forward(self, x: torch.Tensor, tau: float = 0.5,
                hard: bool = False) -> torch.Tensor:
        """Returns (batch, 2): [mean_pred, log_sigma_pred]."""
        h = self.act(self.layer1(x, tau=tau, hard=hard))
        return self.out(h)

    def snapshot_epoch0(self):
        self.layer1.snapshot_epoch0()

    def rewind_to_epoch0(self):
        self.layer1.rewind_to_epoch0()
        self._reset_output_head()

    def elbo(self, x: torch.Tensor, y: torch.Tensor,
             prior_pi1: float, n_data: int,
             kl_weight: float = 1.0, tau: float = 0.5) -> torch.Tensor:
        """
        ELBO for heteroscedastic Gaussian regression (S=1 per paper B.3).

        log p(y | x, θ) = −½ log(2π) − log σ̂ − ½ (y − μ̂)² / σ̂²
        (constant −½ log(2π) omitted — doesn't affect gradients)
        """
        out     = self.forward(x, tau=tau, hard=False)
        mu_hat  = out[:, 0:1]                          # (batch, 1)
        log_sig = out[:, 1:2].clamp(-6.0, 4.0)        # (batch, 1), clipped
        # Gaussian NLL (summed over batch)
        log_like = (-log_sig
                    - 0.5 * (y - mu_hat).pow(2) * torch.exp(-2 * log_sig)
                    ).sum()
        kl = self.layer1.kl_v(prior_pi1) + self.layer1.kl_weights()
        scale = n_data / x.size(0)
        return scale * log_like - kl_weight * kl

    @torch.no_grad()
    def predict(self, x: torch.Tensor,
                n_samples: int = 10) -> tuple[torch.Tensor, torch.Tensor,
                                               torch.Tensor, torch.Tensor]:
        """
        MC predictive distribution via law of total variance.

        Returns
        -------
        mean_pred   : (batch,)  — MC-averaged predicted mean
        total_var   : (batch,)  — total predictive variance (σ²_aleat + σ²_epist)
        aleat_var   : (batch,)  — aleatoric variance E_θ[σ̂²]
        epist_var   : (batch,)  — epistemic variance  Var_θ[μ̂]
        """
        means, vars_ = [], []
        for _ in range(n_samples):
            out = self.forward(x, tau=0.5, hard=False)
            means.append(out[:, 0])
            vars_.append(torch.exp(2 * out[:, 1].clamp(-6.0, 4.0)))

        means = torch.stack(means, dim=0)   # (S, batch)
        vars_ = torch.stack(vars_, dim=0)   # (S, batch)

        mean_pred = means.mean(0)
        aleat_var = vars_.mean(0)                        # E_θ[σ̂²]
        epist_var = means.var(0)                         # Var_θ[μ̂]
        total_var = aleat_var + epist_var                # law of total variance

        return mean_pred, total_var, aleat_var, epist_var


# ─────────────────────────────────────────────────────────────────────────────
# 3.  PRIOR CALIBRATION  (Corollary 4.5, single-layer version)
# ─────────────────────────────────────────────────────────────────────────────

def compute_prior_pi(k_in: int, k_out: int, k_next: int,
                     n: int, L: int,
                     B_l: float | None = None) -> float:
    """
    Layer-wise prior inclusion probability from Corollary 4.5.

    Parameters
    ----------
    k_in   : input units to this layer
    k_out  : units in this layer
    k_next : units in the next layer (output layer for single-hidden)
    n      : training-set size
    L      : total hidden layers (1 for Year)
    B_l    : max L1 norm; defaults to k_out+1 if None
    """
    if B_l is None:
        B_l = k_out + 1
    log_n   = math.log(n)
    pi_val  = min(
        1.0,
        math.sqrt(
            (2 * log_n * k_next)
            / (n * L * k_in * B_l ** 2)
        )
    )
    return float(np.clip(pi_val, 1e-6, 1 - 1e-6))


# ─────────────────────────────────────────────────────────────────────────────
# 4.  SPARSE-NETWORK UTILITIES (adapted for single layer)
# ─────────────────────────────────────────────────────────────────────────────

def magnitude_prune_year(model: NAPNetworkYear,
                          prune_fraction: float,
                          mask: dict | None = None) -> dict:
    """Global unstructured magnitude pruning on layer1.mu only."""
    all_weights, all_names = [], []
    for name, param in model.named_parameters():
        if "mu" in name and "out" not in name:
            w_mag = param.data.abs()
            if mask and name in mask:
                w_mag = w_mag * mask[name].float()
            all_weights.append(w_mag.flatten())
            all_names.append(name)

    flat_all  = torch.cat(all_weights)
    threshold = torch.quantile(flat_all[flat_all > 0], prune_fraction)

    new_mask = {}
    for name, param in model.named_parameters():
        if name in all_names:
            m = param.data.abs() > threshold
            if mask and name in mask:
                m = m & mask[name]
            new_mask[name] = m
        elif mask and name in mask:
            new_mask[name] = mask[name]
    return new_mask


def apply_mask(model: NAPNetworkYear, mask: dict):
    """Zero out pruned weights after each optimiser step."""
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in mask:
                param.data *= mask[name].float()


def count_surviving_nodes_year(model: NAPNetworkYear, mask: dict) -> dict:
    """Count neurons in layer1 with at least one surviving incoming weight."""
    mu_name = "layer1.mu"
    if mu_name in mask:
        alive = mask[mu_name].any(dim=1).sum().item()
    else:
        alive = model.layer1.out_features
    return {"layer1": int(alive)}


def max_l1_norm_year(model: NAPNetworkYear, mask: dict) -> dict:
    """Max L1 weight norm over surviving neurons in layer1."""
    mu  = model.layer1.mu.data
    mn  = "layer1.mu"
    if mn in mask:
        mu = mu * mask[mn].float()
    return {"layer1": float(mu.abs().sum(dim=1).max().item())}


# ─────────────────────────────────────────────────────────────────────────────
# 5.  TRAINING UTILITIES
# ─────────────────────────────────────────────────────────────────────────────

def train_one_epoch_year(model: NAPNetworkYear,
                          loader: DataLoader,
                          optimizer: torch.optim.Optimizer,
                          prior_pi1: float,
                          n_data: int,
                          kl_weight: float = 1.0,
                          tau: float = 0.5,
                          mask: dict | None = None,
                          hard: bool = False) -> float:
    """One epoch of ELBO maximisation (Gaussian NLL + KL)."""
    model.train()
    total_loss = 0.0
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        loss = -model.elbo(x, y, prior_pi1, n_data,
                           kl_weight=kl_weight, tau=tau)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        if mask:
            apply_mask(model, mask)
        total_loss += loss.item()
    return total_loss / len(loader)


# ─────────────────────────────────────────────────────────────────────────────
# 6.  EVALUATION & UNCERTAINTY FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

@torch.no_grad()
def evaluate_rmse(model: NAPNetworkYear, loader: DataLoader,
                  y_std: float, n_samples: int = 10) -> float:
    """RMSE in original year units (back-transformed from normalised scale)."""
    model.eval()
    sq = []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        mu_hat, *_ = model.predict(x, n_samples=n_samples)
        sq.append((mu_hat.unsqueeze(1) - y).pow(2).cpu())
    mse = torch.cat(sq).mean().item()
    return math.sqrt(mse) * y_std   # back-transform to years


@torch.no_grad()
def evaluate_rmse_with_std(model: NAPNetworkYear, loader: DataLoader,
                            y_std: float, n_samples: int = 5) -> tuple[float, float]:
    """
    RMSE ± std across n_samples independent MC forward passes.
    Provides the uncertainty band for training-curve plots.
    Each sample uses a fresh draw of Gumbel gates + weight noise.
    """
    model.eval()
    sample_rmses = []
    for _ in range(n_samples):
        sq = []
        for x, y in loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            out    = model(x, tau=0.5, hard=False)
            mu_hat = out[:, 0:1]
            sq.append((mu_hat - y).pow(2).cpu())
        rmse = math.sqrt(torch.cat(sq).mean().item()) * y_std
        sample_rmses.append(rmse)
    return float(np.mean(sample_rmses)), float(np.std(sample_rmses))


@torch.no_grad()
def compute_regression_uncertainty(model: NAPNetworkYear,
                                    loader: DataLoader,
                                    y_std: float,
                                    n_samples: int = 20
                                    ) -> tuple[np.ndarray, np.ndarray,
                                               np.ndarray, np.ndarray, np.ndarray]:
    """
    Full uncertainty decomposition on the test set (original scale).

    Returns (all in years):
        y_true      : true year values
        y_pred      : predicted mean
        total_std   : sqrt(aleatoric_var + epistemic_var)
        aleat_std   : sqrt(E_θ[σ̂²])   — irreducible noise
        epist_std   : sqrt(Var_θ[μ̂])  — parameter uncertainty
    """
    model.eval()
    yt, yp, ts, als, eps = [], [], [], [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        mean, tot_var, aleat_var, epist_var = model.predict(x, n_samples)

        yt.append(y.squeeze(1).cpu().numpy()         * y_std)
        yp.append(mean.cpu().numpy()                  * y_std)
        ts.append(tot_var.sqrt().cpu().numpy()        * y_std)
        als.append(aleat_var.sqrt().cpu().numpy()     * y_std)
        eps.append(epist_var.sqrt().cpu().numpy()     * y_std)

    return (np.concatenate(yt), np.concatenate(yp),
            np.concatenate(ts), np.concatenate(als), np.concatenate(eps))


def _snapshot_sparsity_year(model: NAPNetworkYear) -> dict:
    """Capture π̃ statistics for layer1 at the current epoch."""
    pi  = model.layer1.pi_tilde.detach().cpu().numpy()
    return {
        "layer1_mean"       : float(pi.mean()),
        "layer1_std"        : float(pi.std()),
        "layer1_frac_active": float((pi > 0.5).mean()),
        "layer1_sparsity"   : float((pi <= 0.5).mean()),
    }


# ─────────────────────────────────────────────────────────────────────────────
# 7.  PHASE 1 — BAYESIAN LOTTERY TICKET (IMP)
# ─────────────────────────────────────────────────────────────────────────────

def run_imp_year(model: NAPNetworkYear,
                 train_loader: DataLoader,
                 test_loader:  DataLoader,
                 n_data: int,
                 y_std: float,
                 imp_rounds: int     = 5,
                 prune_fraction: float = 0.20,
                 epochs_per_round: int = 100,
                 lr: float           = 1e-3,
                 prior_pi1: float    = 0.01) -> tuple[NAPNetworkYear, dict, dict]:
    """
    Phase 1: Iterative Magnitude Pruning on the regression BNN.

    Returns (model_with_ticket_weights, final_mask, history_dict).
    """
    print("\n" + "="*60)
    print("PHASE 1 — Bayesian Lottery Ticket (IMP) — Year MSD")
    print("="*60)

    model.snapshot_epoch0()
    mask = {}
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # per-round history (IMP epochs are short — 100 — log every epoch)
    imp_history: dict = defaultdict(list)

    for rnd in range(1, imp_rounds + 1):
        print(f"\n  IMP round {rnd}/{imp_rounds}")

        for epoch in range(1, epochs_per_round + 1):
            loss = train_one_epoch_year(model, train_loader, optimizer,
                                        prior_pi1, n_data,
                                        kl_weight=1.0, tau=0.5,
                                        mask=mask if mask else None,
                                        hard=True)
            # log every epoch (100 is small enough)
            rmse_m, rmse_s = evaluate_rmse_with_std(model, test_loader,
                                                     y_std, n_samples=3)
            cumulative_epoch = (rnd - 1) * epochs_per_round + epoch
            imp_history["epoch"].append(cumulative_epoch)
            imp_history["round"].append(rnd)
            imp_history["rmse_mean"].append(rmse_m)
            imp_history["rmse_std"].append(rmse_s)

            if epoch % 20 == 0 or epoch == epochs_per_round:
                print(f"    epoch {epoch:3d} | loss {loss:12.2f} "
                      f"| test RMSE {rmse_m:.3f} ± {rmse_s:.3f} yr")

        # prune
        mask     = magnitude_prune_year(model, prune_fraction, mask if mask else None)
        surviving = count_surviving_nodes_year(model, mask)
        print(f"  → surviving nodes  : {surviving}")

        imp_history["surviving_l1"].append(surviving["layer1"])
        # record per-round breakpoints for the bar chart
        imp_history["round_at_prune"].append(rnd)

        # LTH rewind
        model.rewind_to_epoch0()
        apply_mask(model, mask)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    rmse_final = evaluate_rmse(model, test_loader, y_std, n_samples=10)
    print(f"\n[IMP done] winning-ticket RMSE: {rmse_final:.3f} yr")
    return model, mask, dict(imp_history)


# ─────────────────────────────────────────────────────────────────────────────
# 8.  PHASE 2 — SS-IG ON WINNING TICKET (SOFT GATE)
# ─────────────────────────────────────────────────────────────────────────────

def run_ssig_soft_year(model: NAPNetworkYear,
                        train_loader: DataLoader,
                        test_loader:  DataLoader,
                        n_data: int,
                        y_std: float,
                        mask: dict,
                        surviving: dict,
                        b_norms: dict,
                        epochs: int       = 100,
                        lr: float         = 1e-3,
                        kl_anneal_ep: int = 40,
                        L: int            = 1) -> tuple[NAPNetworkYear, dict]:
    """
    Phase 2: Stochastic Subnetwork Identification with soft Gumbel-Softmax.
    Prior π is recalibrated to the winning-ticket dimensions.
    KL weight annealed 0 → 1 over first kl_anneal_ep epochs.

    Returns (model, history_dict).
    """
    print("\n" + "="*60)
    print("PHASE 2 — SS-IG on winning ticket (soft gate) — Year MSD")
    print("="*60)

    s1 = surviving["layer1"]
    b1 = b_norms["layer1"]

    # recalibrate prior to ticket dimensions (output_dim=1 for regression mean)
    pi1 = compute_prior_pi(model.input_dim, s1, 1, n_data, L, B_l=b1)
    print(f"  Ticket-calibrated prior: π₁ = {pi1:.6f}")

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history: dict = defaultdict(list)

    for epoch in range(1, epochs + 1):
        kl_w = min(1.0, epoch / kl_anneal_ep)
        loss = train_one_epoch_year(model, train_loader, optimizer,
                                    pi1, n_data,
                                    kl_weight=kl_w, tau=0.5,
                                    mask=mask, hard=False)

        # every epoch (100 epochs total — log all)
        rmse_m, rmse_s = evaluate_rmse_with_std(model, test_loader,
                                                 y_std, n_samples=3)
        snap = _snapshot_sparsity_year(model)

        history["epoch"].append(epoch)
        history["rmse_mean"].append(rmse_m)
        history["rmse_std"].append(rmse_s)
        for k, v in snap.items():
            history[k].append(v)

        if epoch % 10 == 0 or epoch == epochs:
            pi_l1 = model.layer1.pi_tilde.detach().cpu()
            print(f"  epoch {epoch:4d} | kl_w {kl_w:.2f} | loss {loss:12.2f} "
                  f"| RMSE {rmse_m:.3f} ± {rmse_s:.3f} yr "
                  f"| mean π̃₁={pi_l1.mean():.4f}")

    return model, dict(history)


# ─────────────────────────────────────────────────────────────────────────────
# 9.  PHASE 3 — POISSON-BINOMIAL LAYER-SIZE SELECTION
# ─────────────────────────────────────────────────────────────────────────────

def poisson_binomial_pmf(probs: torch.Tensor) -> np.ndarray:
    """
    Exact Poisson-Binomial PMF via convolution.
    P(K = k | π̃_1, …, π̃_n) using the recursive convolution algorithm.
    """
    n   = len(probs)
    pmf = np.zeros(n + 1)
    pmf[0] = 1.0
    for p in probs.numpy():
        new_pmf       = pmf.copy()
        new_pmf[1:]  += pmf[:-1] * p
        new_pmf[:-1] *= (1 - p)
        pmf = new_pmf
    return pmf


def select_optimal_layer_size_year(model: NAPNetworkYear) -> dict:
    """
    Phase 3: find K^op_1 = argmax_k P(K=k | π̃_1,…,π̃_{K_1}).

    Returns {'layer1': int}
    """
    print("\n" + "="*60)
    print("PHASE 3 — Poisson-Binomial optimal size (Year MSD)")
    print("="*60)

    pi_vec  = model.layer1.pi_tilde.detach().cpu()
    pmf     = poisson_binomial_pmf(pi_vec)
    k_opt   = max(1, int(pmf.argmax()))

    print(f"  layer1: K^op = {k_opt}  (out of {len(pi_vec)})")
    print(f"          P(K={k_opt}) = {pmf[k_opt]:.6f} "
          f"| mean π̃ = {pi_vec.mean():.4f}")
    return {"layer1": k_opt}


# ─────────────────────────────────────────────────────────────────────────────
# 10. PHASE 4 — NEURAL ATOM SELECTION
# ─────────────────────────────────────────────────────────────────────────────

def select_atoms_year(model: NAPNetworkYear, k_optimal: dict) -> dict:
    """
    Phase 4: rank layer1 neurons by π̃_j, keep top K^op_1.
    Returns {'layer1': LongTensor of selected indices}.
    """
    print("\n" + "="*60)
    print("PHASE 4 — Neural atom selection (Year MSD)")
    print("="*60)

    pi_vec  = model.layer1.pi_tilde.detach().cpu()
    k_opt   = k_optimal["layer1"]
    ranked  = torch.argsort(pi_vec, descending=True)
    chosen  = ranked[:k_opt].sort().values

    min_sel = pi_vec[chosen].min().item()
    max_rej = (f"{pi_vec[ranked[k_opt:]].max().item():.4f}"
               if k_opt < len(pi_vec) else "n/a")
    print(f"  layer1: selected {k_opt} atoms "
          f"| min π̃ selected = {min_sel:.4f} "
          f"| max π̃ rejected = {max_rej}")
    return {"layer1": chosen}


def build_sparse_year_network(selected: dict,
                               input_dim: int = 90,
                               sigma0: float  = 1.0) -> NAPNetworkYear:
    """
    Construct a new NAPNetworkYear with hidden = K^op_1 atoms.
    Snapshots a fresh Kaiming init as epoch-0 for Phase-5 rewind.
    """
    k1 = len(selected["layer1"])
    sparse = NAPNetworkYear(input_dim=input_dim, hidden=k1,
                             sigma0=sigma0).to(DEVICE)
    sparse.snapshot_epoch0()
    print(f"\n[sparse] built: {input_dim} → {k1} → 2")
    return sparse


# ─────────────────────────────────────────────────────────────────────────────
# 11. PHASE 5 — FINE-TUNING THE SPARSE BNN
# ─────────────────────────────────────────────────────────────────────────────

def run_finetune_year(sparse_model: NAPNetworkYear,
                      train_loader: DataLoader,
                      test_loader:  DataLoader,
                      n_data: int,
                      y_std: float,
                      k_optimal: dict,
                      b_norms: dict,
                      epochs: int = 100,
                      lr: float   = 1e-3,
                      L: int      = 1) -> tuple[NAPNetworkYear, dict]:
    """
    Phase 5: fine-tune the sparse BNN from epoch-0 weights (LTH-style rewind).
    Prior is recalibrated to the sparse topology K^op_1.

    Returns (sparse_model, history_dict).
    """
    print("\n" + "="*60)
    print("PHASE 5 — Fine-tuning sparse BNN — Year MSD")
    print("="*60)

    sparse_model.rewind_to_epoch0()

    k1 = k_optimal["layer1"]
    b1 = b_norms["layer1"]
    pi1 = compute_prior_pi(sparse_model.input_dim, k1, 1, n_data, L, B_l=b1)
    print(f"  Sparse-topology prior: π₁ = {pi1:.6f}")

    optimizer = torch.optim.Adam(sparse_model.parameters(), lr=lr)
    history: dict = defaultdict(list)

    for epoch in range(1, epochs + 1):
        loss = train_one_epoch_year(sparse_model, train_loader, optimizer,
                                    pi1, n_data,
                                    kl_weight=1.0, tau=0.5,
                                    mask=None, hard=False)

        rmse_m, rmse_s = evaluate_rmse_with_std(sparse_model, test_loader,
                                                 y_std, n_samples=3)
        snap = _snapshot_sparsity_year(sparse_model)

        history["epoch"].append(epoch)
        history["rmse_mean"].append(rmse_m)
        history["rmse_std"].append(rmse_s)
        for k, v in snap.items():
            history[k].append(v)

        if epoch % 10 == 0 or epoch == epochs:
            print(f"  epoch {epoch:4d} | loss {loss:12.2f} "
                  f"| RMSE {rmse_m:.3f} ± {rmse_s:.3f} yr")

    return sparse_model, dict(history)


# ─────────────────────────────────────────────────────────────────────────────
# 12. COMPRESSION REPORT
# ─────────────────────────────────────────────────────────────────────────────

def compression_report_year(input_dim: int, hidden_full: int,
                              k_optimal: dict, surviving: dict):
    k1   = k_optimal["layer1"]
    s1   = surviving["layer1"]
    # params: layer1 weights+biases + output layer (2 neurons)
    full_p   = input_dim * hidden_full + hidden_full + hidden_full * 2 + 2
    sparse_p = input_dim * k1          + k1          + k1          * 2 + 2
    print("\n" + "="*60)
    print("COMPRESSION REPORT — Year MSD")
    print("="*60)
    print(f"  Dense  model  : 90 → {hidden_full} → 2   | {full_p:,} params")
    print(f"  IMP ticket    : 90 → {s1} surviving nodes")
    print(f"  NAP sparse    : 90 → {k1} → 2       | {sparse_p:,} params")
    print(f"  Compression   : {full_p / max(1,sparse_p):.1f}×  "
          f"({100*(1-sparse_p/full_p):.1f}% reduction)")
    print("="*60)


# ─────────────────────────────────────────────────────────────────────────────
# 13. DIAGNOSTIC FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

@torch.no_grad()
def zero_input_rmse(model: NAPNetworkYear, loader: DataLoader,
                    y_std: float, n_samples: int = 10) -> float:
    """Feed all-zero inputs — high RMSE confirms the model uses input features."""
    model.eval()
    sq = []
    for x, y in loader:
        x_zeros = torch.zeros_like(x).to(DEVICE)
        y = y.to(DEVICE)
        mu, *_ = model.predict(x_zeros, n_samples=n_samples)
        sq.append((mu.unsqueeze(1) - y).pow(2).cpu())
    return math.sqrt(torch.cat(sq).mean().item()) * y_std


@torch.no_grad()
def shuffled_label_rmse(model: NAPNetworkYear, loader: DataLoader,
                         y_std: float, n_samples: int = 10) -> float:
    """Feed shuffled labels — high RMSE confirms the model learned true structure."""
    model.eval()
    sq = []
    for x, y in loader:
        x = x.to(DEVICE)
        y_shuf = y[torch.randperm(y.size(0))].to(DEVICE)
        mu, *_ = model.predict(x, n_samples=n_samples)
        sq.append((mu.unsqueeze(1) - y_shuf).pow(2).cpu())
    return math.sqrt(torch.cat(sq).mean().item()) * y_std


# ─────────────────────────────────────────────────────────────────────────────
# 14. PLOTS — 6-PANEL UNCERTAINTY-AWARE FIGURE (regression version)
# ─────────────────────────────────────────────────────────────────────────────

_C = {
    "imp"      : "#E07B39",
    "ssig"     : "#4C72B0",
    "ft"       : "#27AE60",
    "l1"       : "#8E44AD",
    "aleat"    : "#E67E22",
    "epist"    : "#2980B9",
    "total"    : "#7F8C8D",
    "bg"       : "#F8F9FA",
    "grid"     : "#DEE2E6",
}


def plot_all_curves_year(imp_history: dict,
                          ssig_history: dict,
                          ft_history: dict,
                          uq_data: tuple,
                          y_std: float,
                          dataset_name: str = "year_msd",
                          save_path: str    = "nap_year_training_curves.png"):
    """
    6-panel uncertainty-aware training figure for regression:

      [0,0] A — RMSE vs Epoch (SS-IG, every epoch, ±1σ MC band)
      [0,1] B — RMSE vs Epoch (Fine-tune, every epoch, ±1σ MC band)
      [1,0] C — IMP: surviving nodes per round (bar chart)
      [1,1] D — Node sparsity (π̃ active fraction) vs epoch (SS-IG + FT)
      [2,0] E — Predictive uncertainty: total/aleatoric/epistemic std distributions
      [2,1] F — Regression reliability diagram (coverage calibration)
    """
    y_true, y_pred, tot_std, aleat_std, epist_std = uq_data

    fig, axes = plt.subplots(3, 2, figsize=(16, 18), facecolor="white")
    ax_ssig, ax_ft   = axes[0, 0], axes[0, 1]
    ax_imp,  ax_spar = axes[1, 0], axes[1, 1]
    ax_uq,   ax_rel  = axes[2, 0], axes[2, 1]

    def _shade(ax, x, mean, std, color, alpha=0.20):
        mean, std = np.asarray(mean), np.asarray(std)
        ax.fill_between(x, mean - std, mean + std,
                        alpha=alpha, color=color, linewidth=0)

    def _style(ax, xlabel="", ylabel="", title=""):
        ax.set_facecolor(_C["bg"])
        ax.grid(True, color=_C["grid"], linewidth=0.7, linestyle="--", zorder=0)
        ax.spines[["top", "right"]].set_visible(False)
        if xlabel: ax.set_xlabel(xlabel, fontsize=10)
        if ylabel: ax.set_ylabel(ylabel, fontsize=10)
        if title:  ax.set_title(title, fontsize=11, fontweight="bold", pad=8)

    def _rmse_panel(ax, hist, color, title, kl_end=None):
        _style(ax, xlabel="Epoch", ylabel="Test RMSE (years)", title=title)
        ep  = np.asarray(hist["epoch"])
        mu  = np.asarray(hist["rmse_mean"])
        std = np.asarray(hist["rmse_std"])
        # every-epoch scatter
        ax.scatter(ep, mu, s=6, alpha=0.4, color=color,
                   linewidths=0, zorder=3, label="RMSE per epoch")
        _shade(ax, ep, mu, std, color, alpha=0.22)
        # moving-average trend line
        w   = max(1, len(ep) // 20)
        ma  = np.convolve(mu, np.ones(w)/w, mode="same")
        ax.plot(ep, ma, color=color, linewidth=2.0, alpha=0.9,
                zorder=4, label=f"Moving avg (w={w})")
        ax.set_ylim(max(0, mu.min() - 1), mu.max() + 1)
        if kl_end is not None:
            ax.axvline(kl_end, color="#AAA", linewidth=1.1, linestyle=":")
            ax.text(kl_end + ep.max()*0.01,
                    ax.get_ylim()[0] + 0.5, "KL=1", fontsize=7, color="#888")
        ax.legend(fontsize=8, loc="upper right")

    # ── Panel A: SS-IG RMSE ──────────────────────────────────────────────────
    _rmse_panel(ax_ssig, ssig_history, _C["ssig"],
                "A — RMSE vs Epoch · Phase 2 SS-IG  (every epoch · ±1σ)",
                kl_end=min(40, len(ssig_history["epoch"])))

    # ── Panel B: Fine-tune RMSE ───────────────────────────────────────────────
    _rmse_panel(ax_ft, ft_history, _C["ft"],
                "B — RMSE vs Epoch · Phase 5 Fine-tune  (every epoch · ±1σ)")

    # ── Panel C: IMP surviving nodes ─────────────────────────────────────────
    _style(ax_imp, xlabel="IMP Round", ylabel="Surviving Nodes",
           title="C — Surviving Nodes per IMP Round")

    # one bar per round (from round_at_prune list)
    rnd_ticks = np.asarray(imp_history.get("round_at_prune",
                            list(range(1, len(imp_history["surviving_l1"])+1))))
    s1_vals   = np.asarray(imp_history["surviving_l1"])
    ax_imp.bar(rnd_ticks, s1_vals, color=_C["l1"], alpha=0.85, zorder=3,
               label="Layer 1")
    ax_imp.set_xticks(rnd_ticks)
    for xi, yi in zip(rnd_ticks, s1_vals):
        ax_imp.text(xi, yi + 0.5, str(int(yi)),
                    ha="center", va="bottom", fontsize=8, color=_C["l1"])
    ax_imp.legend(fontsize=9)

    # ── Panel D: Node sparsity vs Epoch (SS-IG + FT) ─────────────────────────
    _style(ax_spar,
           xlabel="Epoch", ylabel="Fraction of Neurons Active  (π̃ > 0.5)",
           title="D — Node Sparsity vs Epoch (± 1σ from π̃ spread)")

    for hist, col, label in [
        (ssig_history, _C["ssig"], "SS-IG L1"),
        (ft_history,   _C["ft"],   "FT L1"),
    ]:
        ep = np.asarray(hist["epoch"])
        fa = np.asarray(hist["layer1_frac_active"])
        sd = np.asarray(hist["layer1_std"])
        ax_spar.plot(ep, fa, color=col, linewidth=2.0, label=label, zorder=3)
        _shade(ax_spar, ep, fa, sd, col)

    ax_spar.set_ylim(-0.05, 1.05)
    ax_spar.legend(fontsize=9, loc="upper right")
    ax_spar.axvline(ssig_history["epoch"][-1], color="#AAA",
                    linewidth=1, linestyle=":", zorder=1)
    ax_spar.text(ssig_history["epoch"][-1] * 0.97, 0.03,
                 "← SS-IG | FT →", fontsize=7, color="#777", ha="right")

    # ── Panel E: Predictive uncertainty distributions ─────────────────────────
    _style(ax_uq,
           xlabel="Uncertainty  (years)",
           ylabel="Density",
           title="E — Predictive Uncertainty Decomposition (Final Model)")

    bins = np.linspace(0, np.percentile(tot_std, 99) * 1.05, 60)
    ax_uq.hist(tot_std,   bins=bins, density=True, alpha=0.55,
               color=_C["total"], label="Total σ",    zorder=3)
    ax_uq.hist(aleat_std, bins=bins, density=True, alpha=0.55,
               color=_C["aleat"], label="Aleatoric σ", zorder=3)
    ax_uq.hist(epist_std, bins=bins, density=True, alpha=0.55,
               color=_C["epist"], label="Epistemic σ", zorder=3)
    ax_uq.legend(fontsize=9)

    # inset: scatter of epistemic vs aleatoric
    ax_in = ax_uq.inset_axes([0.55, 0.40, 0.42, 0.55])
    ax_in.set_facecolor(_C["bg"])
    rng = np.random.default_rng(42)
    idx = rng.choice(len(tot_std), size=min(2000, len(tot_std)), replace=False)
    ax_in.scatter(aleat_std[idx], epist_std[idx],
                  c=_C["epist"], alpha=0.25, s=4, linewidths=0)
    ax_in.set_xlabel("Aleatoric σ (yr)", fontsize=7)
    ax_in.set_ylabel("Epistemic σ (yr)", fontsize=7)
    ax_in.set_title("Epist. vs Aleat.", fontsize=7)
    ax_in.tick_params(labelsize=6)
    ax_in.spines[["top", "right"]].set_visible(False)

    ax_uq.text(0.02, 0.97,
               (f"Mean total σ     = {tot_std.mean():.3f} yr\n"
                f"Mean aleatoric σ = {aleat_std.mean():.3f} yr\n"
                f"Mean epistemic σ = {epist_std.mean():.3f} yr\n"
                f"Epist. / Total   = {epist_std.mean()/tot_std.mean():.3f}"),
               transform=ax_uq.transAxes, fontsize=8, verticalalignment="top",
               bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                         edgecolor="#CCC", alpha=0.9))

    # ── Panel F: Regression reliability (coverage) diagram ───────────────────
    _style(ax_rel,
           xlabel="Nominal Coverage  (1 − α)",
           ylabel="Observed Coverage",
           title="F — Regression Reliability Diagram (Gaussian PI Coverage)")

    # Gaussian prediction intervals at multiple confidence levels
    alphas   = np.linspace(0.02, 0.99, 50)
    observed = []
    for a in alphas:
        z    = scipy_stats.norm.ppf(1 - a/2)    # two-sided z for level (1-α)
        lo   = y_pred - z * tot_std
        hi   = y_pred + z * tot_std
        cov  = float(np.mean((y_true >= lo) & (y_true <= hi)))
        observed.append(cov)
    observed = np.array(observed)
    nominal  = 1 - alphas

    ax_rel.plot([0, 1], [0, 1], "k--", linewidth=1.3,
                label="Perfect calibration", zorder=4)
    ax_rel.fill_between(nominal, nominal, observed,
                        where=(observed > nominal),
                        alpha=0.20, color="#2ECC71", label="Overconfident zone")
    ax_rel.fill_between(nominal, nominal, observed,
                        where=(observed < nominal),
                        alpha=0.20, color="#E74C3C", label="Underconfident zone")
    ax_rel.plot(nominal, observed, color=_C["ssig"],
                linewidth=2.2, zorder=3, label="Model coverage")

    # ECE for regression: mean |observed - nominal|
    cal_err = float(np.mean(np.abs(observed - nominal)))
    ax_rel.text(0.04, 0.93,
                f"Calibration Error\n(mean |obs−nom|) = {cal_err:.4f}",
                transform=ax_rel.transAxes, fontsize=9, verticalalignment="top",
                bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                          edgecolor="#CCC", alpha=0.9))
    ax_rel.set_xlim(0, 1); ax_rel.set_ylim(0, 1)
    ax_rel.legend(fontsize=8, loc="lower right")

    # ── suptitle & save ───────────────────────────────────────────────────────
    fig.suptitle(
        f"Neural Atom Prevalence (NAP) — {dataset_name.upper()}\n"
        f"Uncertainty-Aware Training Diagnostics  (Regression)",
        fontsize=14, fontweight="bold", y=0.995
    )
    plt.tight_layout(rect=[0, 0, 1, 0.99])
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"\n[plot] saved → {save_path}")
    return save_path


# ─────────────────────────────────────────────────────────────────────────────
# 15. MAIN PIPELINE
# ─────────────────────────────────────────────────────────────────────────────

def run_nap_year_pipeline(data_path: str):
    """
    Full 5-phase NAP pipeline for YearPredictionMSD.

    Parameters
    ----------
    data_path : str
        Path to the local YearPredictionMSD.txt file
        (UCI format: first column = year, next 90 = features).
    """
    # ── hyperparameters — paper Appendix B.2 / B.3 ───────────────────────────
    INPUT_DIM    = 90
    HIDDEN       = 100      # one hidden layer, 100 neurons
    BATCH_SIZE   = 256      # paper B.3
    LR           = 1e-3     # paper B.3
    SIGMA0       = 1.0
    MC_SAMPLES   = 10       # for evaluation (S=1 only during training)
    KL_ANNEAL_EP = 40       # anneal over first 40% of 100 epochs
    L            = 1        # one hidden layer

    # IMP
    IMP_ROUNDS       = 5
    PRUNE_FRAC       = 0.20
    EPOCHS_IMP       = 100   # paper B.3
    PRIOR_PI1_IMP    = 0.01  # dense-architecture prior for IMP

    # SS-IG + fine-tune
    EPOCHS_SSIG = 100        # paper B.3
    EPOCHS_FT   = 100        # paper B.3

    # ── data ─────────────────────────────────────────────────────────────────
    train_loader, test_loader, y_mean, y_std, X_mean, X_std = get_year_loaders(
        data_path, batch_size=BATCH_SIZE
    )
    n_data = len(train_loader.dataset)

    # ── model ────────────────────────────────────────────────────────────────
    model = NAPNetworkYear(input_dim=INPUT_DIM,
                            hidden=HIDDEN,
                            sigma0=SIGMA0).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"\n[model] {n_params:,} parameters on {DEVICE}")
    print(f"[model] architecture: {INPUT_DIM} → {HIDDEN} → 2 (μ̂, log σ̂)")

    # ── Phase 1: IMP ─────────────────────────────────────────────────────────
    model, mask, imp_history = run_imp_year(
        model, train_loader, test_loader, n_data, y_std,
        imp_rounds       = IMP_ROUNDS,
        prune_fraction   = PRUNE_FRAC,
        epochs_per_round = EPOCHS_IMP,
        lr               = LR,
        prior_pi1        = PRIOR_PI1_IMP,
    )

    surviving = count_surviving_nodes_year(model, mask)
    b_norms   = max_l1_norm_year(model, mask)
    print(f"\n[ticket] surviving nodes : {surviving}")
    print(f"[ticket] max L1 norms    : {b_norms}")

    # ── Phase 2: SS-IG soft on ticket ────────────────────────────────────────
    model, ssig_history = run_ssig_soft_year(
        model, train_loader, test_loader, n_data, y_std,
        mask         = mask,
        surviving    = surviving,
        b_norms      = b_norms,
        epochs       = EPOCHS_SSIG,
        lr           = LR,
        kl_anneal_ep = KL_ANNEAL_EP,
        L            = L,
    )

    # ── Phase 3: Poisson-Binomial size selection ──────────────────────────────
    k_optimal = select_optimal_layer_size_year(model)

    # ── Phase 4: atom selection ───────────────────────────────────────────────
    selected = select_atoms_year(model, k_optimal)

    # ── Phase 5: build sparse model and fine-tune ─────────────────────────────
    sparse_model = build_sparse_year_network(selected,
                                              input_dim=INPUT_DIM,
                                              sigma0=SIGMA0)
    sparse_model, ft_history = run_finetune_year(
        sparse_model, train_loader, test_loader, n_data, y_std,
        k_optimal = k_optimal,
        b_norms   = b_norms,
        epochs    = EPOCHS_FT,
        lr        = LR,
        L         = L,
    )

    # ── Final evaluation ──────────────────────────────────────────────────────
    print("\n" + "="*60)
    print("FINAL EVALUATION")
    print("="*60)
    final_rmse = evaluate_rmse(sparse_model, test_loader,
                               y_std, n_samples=MC_SAMPLES)
    print(f"  NAP sparse model test RMSE : {final_rmse:.3f} years")

    # ── Data-use sanity checks ────────────────────────────────────────────────
    zero_r = zero_input_rmse(sparse_model, test_loader, y_std)
    shuf_r = shuffled_label_rmse(sparse_model, test_loader, y_std)
    print(f"\n  Zero-input RMSE    : {zero_r:.3f} yr  (should be >> {final_rmse:.3f})")
    print(f"  Shuffled-label RMSE: {shuf_r:.3f} yr  (should be >> {final_rmse:.3f})")
    print(f"  Data-use gain      : {zero_r - final_rmse:+.3f} yr below zero-input baseline")

    # ── Predictive uncertainty decomposition ──────────────────────────────────
    print("\n[UQ] computing predictive uncertainty on test set …")
    uq_data = compute_regression_uncertainty(sparse_model, test_loader,
                                              y_std, n_samples=20)
    y_true, y_pred, tot_std, aleat_std, epist_std = uq_data
    print(f"  Mean total σ     : {tot_std.mean():.3f} yr")
    print(f"  Mean aleatoric σ : {aleat_std.mean():.3f} yr")
    print(f"  Mean epistemic σ : {epist_std.mean():.3f} yr")
    print(f"  Epist. / Total   : {epist_std.mean() / tot_std.mean():.3f}")

    compression_report_year(INPUT_DIM, HIDDEN, k_optimal, surviving)

    # ── Plots ─────────────────────────────────────────────────────────────────
    plot_path = "nap_year_training_curves.png"
    plot_all_curves_year(imp_history, ssig_history, ft_history,
                          uq_data, y_std,
                          dataset_name="YearPredictionMSD",
                          save_path=plot_path)

    # ── Checkpoint ───────────────────────────────────────────────────────────
    ckpt_path = "nap_year_sparse_model.pt"
    torch.save({
        "state_dict"    : sparse_model.state_dict(),
        "k_optimal"     : k_optimal,
        "surviving"     : surviving,
        "b_norms"       : b_norms,
        "final_rmse"    : final_rmse,
        "y_mean"        : y_mean,
        "y_std"         : y_std,
        "selected"      : {k: v.tolist() for k, v in selected.items()},
        "imp_history"   : imp_history,
        "ssig_history"  : ssig_history,
        "ft_history"    : ft_history,
        "uq_summary"    : {
            "mean_total_std"  : float(tot_std.mean()),
            "mean_aleat_std"  : float(aleat_std.mean()),
            "mean_epist_std"  : float(epist_std.mean()),
            "final_rmse"      : float(final_rmse),
            "zero_input_rmse" : float(zero_r),
        },
    }, ckpt_path)
    print(f"\n[saved] checkpoint → {ckpt_path}")
    print(f"[saved] plot       → {plot_path}")

    return sparse_model, k_optimal, final_rmse


# ─────────────────────────────────────────────────────────────────────────────
# 16. ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────

# ── Set the path to your local YearPredictionMSD file here ───────────────────
DATA_PATH = "/home/student25/Documents/Essaie/Datasetforthesis/yearpredictionmsd/YearPredictionMSD.txt"   
# ─────────────────────────────────────────────────────────────────────────────

import sys as _sys

if __name__ == "__main__":
    import argparse as _ap
    parser = _ap.ArgumentParser(description="NAP pipeline — YearPredictionMSD")
    parser.add_argument("--data", type=str, default=DATA_PATH,
                        help="Path to YearPredictionMSD.txt")
    args, _ = parser.parse_known_args()

    print(f"\n{'='*60}")
    print(f"  Neural Atom Prevalence — YearPredictionMSD (Regression)")
    print(f"  Device  : {DEVICE}")
    print(f"  Workers : {N_WORKERS}")
    print(f"{'='*60}\n")

    run_nap_year_pipeline(data_path=args.data)

# ── In Jupyter, run: ─────────────────────────────────────────────────────────
# run_nap_year_pipeline(data_path="./YearPredictionMSD.txt")
# ─────────────────────────────────────────────────────────────────────────────


  Neural Atom Prevalence — YearPredictionMSD (Regression)
  Device  : cuda
  Workers : 4

[data] loading YearPredictionMSD from: /home/student25/Documents/Essaie/Datasetforthesis/yearpredictionmsd/YearPredictionMSD.txt
[data] raw shape: (515345, 91)
[data] train=463,810  test=51,535 | batch=256 | device=cuda
[data] year range: 1922–2011 | y_mean=1998.40  y_std=10.93
[data] features: 90

[model] 18,502 parameters on cuda
[model] architecture: 90 → 100 → 2 (μ̂, log σ̂)

PHASE 1 — Bayesian Lottery Ticket (IMP) — Year MSD

  IMP round 1/5
    epoch  20 | loss     53391.90 | test RMSE 8.834 ± 0.001 yr
    epoch  40 | loss     40686.74 | test RMSE 8.835 ± 0.001 yr
    epoch  60 | loss     36254.82 | test RMSE 8.794 ± 0.001 yr
    epoch  80 | loss     34229.04 | test RMSE 8.765 ± 0.004 yr
    epoch 100 | loss     43098.90 | test RMSE 8.773 ± 0.005 yr
  → surviving nodes  : {'layer1': 100}

  IMP round 2/5
    epoch  20 | loss     52345.99 | test RMSE 8.878 ± 0.001 yr
    epoch  40 | loss    